# 05 Train YOLO Models

Purpose: train selected YOLO-family object detection models after the smoke test has passed.

**Inputs**

- Prepared YOLO detection dataset
- `configs/experiments.yaml`
- Local model weights, or explicit permission to download weights

**Outputs**

- `outputs/experiments/<experiment_name>/config_used.yaml`
- `outputs/experiments/<experiment_name>/train_log.csv`
- `outputs/experiments/<experiment_name>/metrics.json`
- `outputs/experiments/<experiment_name>/weights/`

Full training can take substantial time. No training starts automatically when this notebook opens.


In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "scripts").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Could not find repo root. Open the notebook from inside the fishDetect repository.")

REPO = find_repo_root()
os.chdir(REPO)

DATASET_ROOT = Path(os.environ.get(
    "FISHDETECT_DATASET_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/merged_viame_v2"
))

PREPARED_ROOT = Path(os.environ.get(
    "FISHDETECT_PREPARED_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/prepared_ml"
))

os.environ["FISHDETECT_DATASET_ROOT"] = str(DATASET_ROOT)
os.environ["FISHDETECT_PREPARED_ROOT"] = str(PREPARED_ROOT)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

print("Repo root:", REPO)
print("Dataset root:", DATASET_ROOT)
print("Prepared root:", PREPARED_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())
print("Prepared parent exists:", PREPARED_ROOT.parent.exists())


## Select Training Scope

Available groups are `smoke`, `first_pass`, `main_comparison`, `capacity_check`, and `extended`. The recommended default after smoke testing is `first_pass`; do not start with `extended`.


In [ ]:
GROUP = "first_pass"
RUN_TRAINING = False
ALLOW_DOWNLOADS = False

print("Selected group:", GROUP)
print("Run training:", RUN_TRAINING)
print("Allow remote weight downloads:", ALLOW_DOWNLOADS)


## Review Configured Experiments


In [ ]:
from fishdetect.config import load_config, get_experiment_group, get_experiment
import pandas as pd

config = load_config("configs/experiments.yaml")
selected = get_experiment_group(config, GROUP)
rows = []
for name in selected:
    exp = get_experiment(config, name)
    rows.append({k: exp.get(k) for k in ["name", "family", "model", "task", "optional"]})
pd.DataFrame(rows)


## Run Selected Group


In [ ]:
if RUN_TRAINING:
    download_flag = "--allow-downloads" if ALLOW_DOWNLOADS else ""
    !python scripts/run_all_experiments.py --config configs/experiments.yaml --group $GROUP --run-full $download_flag
else:
    print("Training not launched. Review GROUP, set RUN_TRAINING = True, and rerun this cell when ready.")


## Train One Model Example


In [ ]:
EXPERIMENT = "yolo8s_det"
RUN_SINGLE_EXPERIMENT = False

if RUN_SINGLE_EXPERIMENT:
    download_flag = "--allow-downloads" if ALLOW_DOWNLOADS else ""
    !python scripts/train_yolo.py --experiment $EXPERIMENT --config configs/experiments.yaml $download_flag
else:
    print("Single-model training not launched. Set RUN_SINGLE_EXPERIMENT = True when ready.")


YOLOv8, YOLOv9, YOLOv10, YOLO11, YOLOv12, and RT-DETR are configured where supported by the local Ultralytics installation. Unsupported or unavailable models are skipped gracefully and recorded in each experiment's `metrics.json`.
